In [1]:
import pandas as pd
import math
import numpy as np
from sqlalchemy import create_engine

In [2]:
database_file = r"C:\Work\imWEBs\test\Garvey Glenn\data\Hydroclimate.db3"
engine = create_engine(f'sqlite:///{database_file}')  
df = pd.read_sql(f"select distinct station from station_data_pcp", engine)
df.head()

,STATION
0,1
1,2
2,3
3,4
4,5


In [3]:
df["STATION"].to_list()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

In [5]:
df = pd.read_sql(f"select min(datetime(date)) start_date from station_data_hmd", engine)
df.head()


,start_date
0,1970-01-01 00:00:00


In [9]:
df['start_date'] = pd.to_datetime(df['start_date'])
start_date = pd.to_datetime(df['start_date'].to_list()[0])
start_date

Timestamp('1970-01-01 00:00:00')

In [2]:
database_file = r"C:\Work\imWEBs\STC\model\STC-Catchabasin-Reservoir-Separated\Model01-SubArea\BMP.db3"
engine = create_engine(f'sqlite:///{database_file}')  

In [6]:
def read_table(table_name:str)->pd.DataFrame:
    """read the whole table and return dataframe"""
    return pd.read_sql(f"select * from {table_name}", engine)

In [7]:
start_year = 2009

In [23]:
grazing_management_df = read_table("GRAMG_management")
field_info_df = read_table("field_info")
livestock_parameter = read_table("livestock_parameter")
fertilizer_parameter = read_table("fertilizer_parameter")

#inner join grazing_management, field_info and livestock_parameter
reach_deposit_df = grazing_management_df[grazing_management_df["Source"] == 1]
reach_deposit_df = reach_deposit_df[reach_deposit_df["Access"] != 2]
reach_deposit_df = reach_deposit_df.merge(field_info_df, left_on="Location", right_index=True, how="inner")
reach_deposit_df = reach_deposit_df.merge(livestock_parameter, left_on="Ani_ID", right_index=True, how="inner")
reach_deposit_df = reach_deposit_df.merge(fertilizer_parameter, left_on="Man_ID", right_index=True, how="inner")

#calcualte manure and drinkwater
reach_deposit_df["Manure"] = reach_deposit_df["GR_Density"] * reach_deposit_df["Area_Ha"] * reach_deposit_df["Ani_Weight"] * reach_deposit_df["Ani_adult"] / 100 / 1000 * reach_deposit_df["Man_Day"] * reach_deposit_df["DayFra"] * reach_deposit_df["Drinking_time"]
reach_deposit_df["DrinkWater"] = reach_deposit_df["GR_Density"] * reach_deposit_df["Area_Ha"] * reach_deposit_df["Ani_Weight"] * reach_deposit_df["Ani_adult"] / 100 * reach_deposit_df["Water_Drink"] / 100 / 1000 

#further calculation and rename
reach_deposit_df["Year"] = reach_deposit_df["Year"] + start_year - 1
reach_deposit_df["Month"] = reach_deposit_df["GraMon"]
reach_deposit_df["Day"] = reach_deposit_df["GraDay"]
reach_deposit_df["StartDate"] = pd.to_datetime(reach_deposit_df[["Year","Month","Day"]])
reach_deposit_df["ReachId"] = reach_deposit_df["SourceID"]
reach_deposit_df["TSS"] = reach_deposit_df["Man_TSS_Fra"] / 100 * reach_deposit_df["Manure"]
reach_deposit_df["NO3"] = reach_deposit_df["FMINN"]  * reach_deposit_df["Manure"]
reach_deposit_df["NH4"] = 0
reach_deposit_df["OrgN"] = reach_deposit_df["FORGN"] * reach_deposit_df["Manure"]
reach_deposit_df["SolP"] = reach_deposit_df["FMINP"] * reach_deposit_df["Manure"]
reach_deposit_df["OrgP"] = reach_deposit_df["FORGP"] * reach_deposit_df["Manure"]

#just keep the useful columns
reach_deposit_df = reach_deposit_df[["ReachId","StartDate","Days","Manure","TSS","NO3","NH4","OrgN","SolP","OrgP","DrinkWater","BankK_Change"]]

#repeat per the days column
expanded_reach_deposit_dfs = []
for index in range(len(reach_deposit_df)):
    #repeat the records based on the number of days and set the date
    repeated_df = pd.DataFrame(np.repeat(reach_deposit_df.iloc[[index]].values,reach_deposit_df.iloc[index].Days,axis=0),columns=reach_deposit_df.columns).reset_index()
    temp = repeated_df["index"].apply(lambda x: pd.Timedelta(x, unit='D'))
    repeated_df["Date"] = repeated_df["StartDate"] + temp
    expanded_reach_deposit_dfs.append(repeated_df)

#concat
reach_deposit_df = pd.concat(expanded_reach_deposit_dfs)

#aggregate for each and date
reach_deposit_df = pd.pivot_table(reach_deposit_df, 
                index = ["ReachId","Date"], 
                values =["Manure","TSS","NO3","NH4","OrgN","SolP","OrgP","DrinkWater","BankK_Change"],
                aggfunc={"Manure":"sum",
                        "TSS":"sum",
                        "NO3":"sum",
                        "NH4":"sum",
                        "OrgN":"sum",
                        "SolP":"sum",
                        "OrgP":"sum",
                        "DrinkWater":"sum",
                        "BankK_Change":"min"})

reach_deposit_df.reset_index(inplace=True)

#get year, month and day
reach_deposit_df["Year"] = reach_deposit_df["Date"].dt.year - start_year + 1
reach_deposit_df["Month"] = reach_deposit_df["Date"].dt.month
reach_deposit_df["Day"] = reach_deposit_df["Date"].dt.day

reach_deposit_df = reach_deposit_df[["ReachId","Year","Month","Day","Manure","TSS","NO3","NH4","OrgN","SolP","OrgP","DrinkWater","BankK_Change"]]

In [25]:
reach_deposit_df.sort_values(["ReachId","Year","Month","Day"],inplace=True)

In [26]:
reach_deposit_df.head()

,ReachId,Year,Month,Day,Manure,TSS,NO3,NH4,OrgN,SolP,OrgP,DrinkWater,BankK_Change
0,2,19,5,1,3.307392,0.826848,0.07607,0,0.095914,0.019844,0.023152,0.064152,0.3
1,2,19,5,2,3.307392,0.826848,0.07607,0,0.095914,0.019844,0.023152,0.064152,0.3
2,2,19,5,3,3.307392,0.826848,0.07607,0,0.095914,0.019844,0.023152,0.064152,0.3
3,2,19,5,4,3.307392,0.826848,0.07607,0,0.095914,0.019844,0.023152,0.064152,0.3
4,2,19,5,5,3.307392,0.826848,0.07607,0,0.095914,0.019844,0.023152,0.064152,0.3


In [27]:
reach_deposit_df.to_csv(r"C:\Work\imWEBs\STC\model\STC-Catchabasin-Reservoir-Separated\Model01-SubArea\reach_deposit.csv")